# sympy-editor in Jupyter

Click-to-edit SymPy expressions as a notebook widget.  Requires `pip install "sympy-editor[jupyter]"` (adds [anywidget](https://anywidget.dev)); works in JupyterLab, Notebook 7, VS Code and Colab.

In [ ]:
from sympy import *
from sympy_editor import edit, register_op, display_html

x, y, n = symbols("x y n")
f = Function("f")

## Basic use

Display the widget, then in the rendering:

* **click** a sub-expression to select it (click the same spot again to select the enclosing expression, or use ↑ ↓ ← →),
* **type** to replace the selection (SymPy syntax) and press **Enter** — the field appears right inside the formula,
* **Enter** or **double-click** edits the selection's current text, **Esc** cancels, **Del** removes the selection from its parent,
* pick a transformation in the dropdown and press **Apply** to apply it to the selection only,
* **Ctrl+Z / Ctrl+Shift+Z** undo and redo.

In [ ]:
w = edit(Integral(exp(-x**2 / 2) / sqrt(2 * pi), (x, -oo, y)) + Sum(f(n) / n**2, (n, 1, oo)) - sin(x) / (x + 1))
w

The widget is live: `w.expr` is always the current expression, so the rest of the notebook can use what you edited by hand.

In [ ]:
w.expr

In [ ]:
simplify(w.expr)

## Reacting to edits

Register a callback to run every time the expression changes in the browser (e.g. to update a plot or a second output).

In [ ]:
from IPython.display import display

log = []
w.on_change(lambda e: log.append(e))

log  # edit something above, then re-run this cell

## Driving the widget from Python

Setting `w.expr` (or calling methods on `w.document`) updates the rendering.

In [ ]:
w.expr = (x + 1)**3 - x**3
w

In [ ]:
w.document.apply("/", "expand")   # same as choosing Expand + Apply with nothing selected
w.refresh()
w.expr

## Custom transformations

Anything registered with `register_op` shows up in the dropdown of widgets created afterwards.

In [ ]:
@register_op("complete_square", label="Complete the square (in x)")
def complete_square(e):
    a, b, c = Poly(e, x).all_coeffs()
    return a * (x + b / (2 * a))**2 + c - b**2 / (4 * a)

edit(2*x**2 + 12*x + 5)

## Symbols keep their assumptions

Typed input is parsed in the context of the expression: existing symbols (with their assumptions) and undefined functions are reused, new names become plain symbols.

In [ ]:
p = Symbol("p", positive=True)
w2 = edit(sqrt(p**2) + f(p))   # try replacing p by p**3: sqrt(p**6) simplifies to p**3 because p is positive
w2

## Without the kernel: static HTML output

`display_html` renders the same editor as plain HTML (no `anywidget` needed).  It also survives `nbconvert --to html`: editing then runs in the browser with Pyodide, but the result is **not** sent back to the kernel.

In [ ]:
display_html(x**2 / y - sin(x))